## 1. Install and import libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from scipy.stats import binomtest

RANDOM_STATE = 42

## 2. Load and inspect the dataset

In [2]:
DATA_PATH = "heart_disease.csv"
TARGET = "target"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn data types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution:")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))

Dataset shape: (25, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1



Column data types:


,dtype
age,int64
sex,int64
cp,int64
trestbps,int64
chol,int64
fbs,int64
restecg,int64
thalach,int64
exang,int64
oldpeak,float64



Missing values:


,missing_count
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0



Duplicate rows: 0

Target distribution:


,count
target,
1,16
0,9


## 3. Basic data preparation

In [3]:
df = df.drop_duplicates().dropna(subset=[TARGET]).copy()

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

if set(y.unique()) != {0, 1}:
    raise ValueError(
        f"Expected target values 0 and 1, but found {sorted(y.unique())}. "
        "Update target encoding before continuing."
    )


X = pd.get_dummies(X, drop_first=False)


X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Positive class (1):", int((y == 1).sum()))
print("Negative class (0):", int((y == 0).sum()))

Features shape: (25, 13)
Target shape: (25,)
Positive class (1): 16
Negative class (0): 9


## 4. Define individual classification models

In [4]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=4, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=3)
    ),
    "SVM": make_pipeline(
        StandardScaler(),
        SVC(probability=True, class_weight="balanced", random_state=RANDOM_STATE)
    ),
}

print("Models:", list(models.keys()))

Models: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN', 'SVM']


## 5. Manually calculate evaluation metrics

In [6]:
def manual_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) else 0.0)

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }

## 6. Generate out-of-fold predictions

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_predictions = {}
oof_probabilities = {}
fold_f1_scores = {}

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

for name, model in models.items():
    oof_predictions[name] = cross_val_predict(
        model, X, y, cv=cv, method="predict"
    )
    oof_probabilities[name] = cross_val_predict(
        model, X, y, cv=cv, method="predict_proba"
    )[:, 1]

    cv_result = cross_validate(model, X, y, cv=cv, scoring=scoring)
    fold_f1_scores[name] = cv_result["test_f1"]

print("Out-of-fold predictions generated for all models.")

## 7. Compare all individual models

In [ ]:
results = {}

for name in models:
    results[name] = manual_metrics(y, oof_predictions[name])
    results[name]["CV F1 mean"] = float(np.mean(fold_f1_scores[name]))
    results[name]["CV F1 std"] = float(np.std(fold_f1_scores[name], ddof=1))

individual_results_df = pd.DataFrame(results).T
metric_columns = [
    "Accuracy", "Precision", "Recall", "Specificity", "F1-score",
    "CV F1 mean", "CV F1 std", "TP", "TN", "FP", "FN"
]
display(individual_results_df[metric_columns].round(3).sort_values("F1-score", ascending=False))

## 8. Calculate dynamic model weights

In [ ]:
raw_weights = {}

for name in models:
    m = results[name]
    raw_weights[name] = (
        0.50 * m["F1-score"]
        + 0.30 * m["Recall"]
        + 0.20 * m["Precision"]
    )

weight_total = sum(raw_weights.values())
if weight_total == 0:
    weights = {name: 1 / len(models) for name in models}
else:
    weights = {name: value / weight_total for name, value in raw_weights.items()}

weights_df = pd.DataFrame({
    "Model": list(weights.keys()),
    "Raw score": [raw_weights[name] for name in weights],
    "Normalized weight": [weights[name] for name in weights]
}).sort_values("Normalized weight", ascending=False)

display(weights_df.round(3))

## 9. Manual Hard Voting

In [ ]:
hard_scores = np.zeros(len(y), dtype=float)

for name in models:
    hard_scores += weights[name] * np.asarray(oof_predictions[name])

hard_vote_predictions = (hard_scores >= 0.5).astype(int)
hard_voting_metrics = manual_metrics(y, hard_vote_predictions)

print("Manual Hard Voting metrics:")
display(pd.DataFrame([hard_voting_metrics], index=["Manual Hard Voting"]).round(3))

## 10. Manual Soft Voting

In [ ]:
soft_scores = np.zeros(len(y), dtype=float)

for name in models:
    soft_scores += weights[name] * oof_probabilities[name]

THRESHOLD = 0.50
soft_vote_predictions = (soft_scores >= THRESHOLD).astype(int)
soft_voting_metrics = manual_metrics(y, soft_vote_predictions)

print("Manual Soft Voting metrics:")
display(pd.DataFrame([soft_voting_metrics], index=["Manual Soft Voting"]).round(3))

## 11. Compare all models and voting classifiers

In [ ]:
all_results = dict(results)
all_results["Manual Hard Voting"] = hard_voting_metrics
all_results["Manual Soft Voting"] = soft_voting_metrics

all_results_df = pd.DataFrame(all_results).T
display(all_results_df[metric_columns].round(3).sort_values("F1-score", ascending=False))

## 12. Select the best model with a custom score

In [ ]:
def custom_selection_score(row):
    return (
        0.50 * row["F1-score"]
        + 0.35 * row["Recall"]
        + 0.15 * row["Precision"]
    )

selection_scores = {
    name: custom_selection_score(all_results_df.loc[name])
    for name in all_results_df.index
}

selection_df = pd.DataFrame({
    "Model": list(selection_scores.keys()),
    "Custom score": list(selection_scores.values())
}).sort_values("Custom score", ascending=False)

display(selection_df.round(3))

best_model_name = selection_df.iloc[0]["Model"]
print("Best model according to this custom score:", best_model_name)
print("This is a data-dependent result; check precision, recall, F1 and confusion counts before selecting.")

## 13. Paired significance test for the two best individual models

In [ ]:
individual_names = sorted(
    models.keys(),
    key=lambda name: selection_scores[name],
    reverse=True
)

model_a, model_b = individual_names[:2]
y_array = np.asarray(y)
pred_a = np.asarray(oof_predictions[model_a])
pred_b = np.asarray(oof_predictions[model_b])

a_correct = pred_a == y_array
b_correct = pred_b == y_array

a_only_correct = int(np.sum(a_correct & ~b_correct))
b_only_correct = int(np.sum(~a_correct & b_correct))
discordant = a_only_correct + b_only_correct

if discordant == 0:
    p_value = 1.0
else:
    p_value = binomtest(
        min(a_only_correct, b_only_correct),
        n=discordant,
        p=0.5,
        alternative="two-sided"
    ).pvalue

print(f"Model A: {model_a}")
print(f"Model B: {model_b}")
print("Rows only model A classified correctly:", a_only_correct)
print("Rows only model B classified correctly:", b_only_correct)
print(f"Exact paired-test p-value: {p_value:.4f}")

if p_value < 0.05:
    print("The paired test detects a difference at the 0.05 level.")
else:
    print("No statistically significant difference detected at the 0.05 level.")

print("Caution: with only 25 rows, statistical power is very low; a non-significant result does not prove equivalence.")